In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, classification_report
import joblib

In [2]:
df = pd.read_csv('../data/final_features_logs.csv')

df = df.sort_values('timestamp')

X = df[['spindle_speed_rpm','prev_temperature_c','prev_vibration_mm_s']]
y = df['failure']

split_idx = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("--- Data Split and Scaling Completed ---")
print(f"Training Features Shape: {X_train_scaled.shape}")

--- Data Split and Scaling Completed ---
Training Features Shape: (2396, 3)


In [3]:
log_model=LogisticRegression(class_weight='balanced',random_state=42)
log_model.fit(X_train_scaled, y_train)

y_pred_log=log_model.predict(X_test_scaled)

print("Logistic Regression Confusion Matrix:\n", confusion_matrix(y_test, y_pred_log))
print("\nLogistic Regression Classification report:\n", classification_report(y_test,y_pred_log))

Logistic Regression Confusion Matrix:
 [[317 259]
 [  9  15]]

Logistic Regression Classification report:
               precision    recall  f1-score   support

           0       0.97      0.55      0.70       576
           1       0.05      0.62      0.10        24

    accuracy                           0.55       600
   macro avg       0.51      0.59      0.40       600
weighted avg       0.94      0.55      0.68       600



In [4]:
rf_model=RandomForestClassifier(random_state=42, class_weight={0: 1, 1:50})
rf_model.fit(X_train_scaled, y_train)

y_pred_rf=rf_model.predict(X_test_scaled)

print("Random Forest Confusion Matrix:\n",confusion_matrix(y_test,y_pred_rf))

print("\nRandom Forest Classification report:\n", classification_report(y_test,y_pred_rf))

Random Forest Confusion Matrix:
 [[576   0]
 [ 24   0]]

Random Forest Classification report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98       576
           1       0.00      0.00      0.00        24

    accuracy                           0.96       600
   macro avg       0.48      0.50      0.49       600
weighted avg       0.92      0.96      0.94       600



c:\Users\PC\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\PC\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\PC\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [5]:
neg_class_count=(y_train==0).sum()
pos_class_count=(y_train==1).sum()
scale_weight=neg_class_count/ pos_class_count

xgboost_model=XGBClassifier(
    scale_pos_weight=scale_weight,
    random_state=42, 
    max_depth=3, 
    learning_rate=0.1

    )
xgboost_model.fit(X_train_scaled,y_train)

y_pred_xgboost=xgboost_model.predict(X_test_scaled)

print("Confusion Matrix:\n", confusion_matrix(y_test,y_pred_xgboost))
print("\nClassification report:\n", classification_report(y_test,y_pred_xgboost))

Confusion Matrix:
 [[442 134]
 [ 15   9]]

Classification report:
               precision    recall  f1-score   support

           0       0.97      0.77      0.86       576
           1       0.06      0.38      0.11        24

    accuracy                           0.75       600
   macro avg       0.52      0.57      0.48       600
weighted avg       0.93      0.75      0.83       600



In [6]:
# Saving the champion model and the scaler permanently for deployment
joblib.dump(xgboost_model,'../models/xgboost_model.pkl')
joblib.dump(scaler,'../models/robust_scaler.pkl')


print("---Champion Model and Scaler saved successfully---")


---Champion Model and Scaler saved successfully---
